# Zero Gaussian Curvature Cone Section

### Imports and setup

In [35]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import PlaceholderNet, GeneralNet
from training.optimizers import GaussNewton, GaussNewtonNew
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### True Surface

In [36]:
def sample_boundary(n_points=1200, dtype=torch.float64):
    third = n_points // 3
    t = torch.linspace(0, 2 * torch.pi, third, dtype=dtype)

    z1 = -torch.zeros_like(t)
    y1 = 1.0 * torch.cos(t)
    x1 = 1.0 * torch.sin(t)

    # Left loop in y-z plane (like helicoid end)
    z2 = torch.ones_like(t)
    y2 = 0.4 * torch.cos(t)
    x2 = 0.4 * torch.sin(t)

    loop1 = torch.vstack([x1, y1, z1])
    loop2 = torch.vstack([x2, y2, z2])
    return torch.cat([loop1, loop2], dim=1).T

pts_boundary = sample_boundary(n_points=1000)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

Output()

### Pretraining

In [37]:
model = PlaceholderNet(ks=[3, 32, 32, 1])
# model = GeneralNet(ks=[3, 128, 128, 128 , 128, 1])
model = model.double()
# Generate random training points
num_pretrain_samples = 10000
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)
pts_pretrain = torch.rand(num_pretrain_samples, 3, dtype=torch.float64) * (bounds[:,1] - bounds[:,0]) + bounds[:,0]


def pretrain_loss(model, params, pts):
    inputs = pts.to(dtype=torch.float64)
    x, y = pts[:, 0], pts[:, 1]
    targets = x**2 + y**2 - 1
    preds = model(inputs).squeeze(1)
    return 0.5 * (preds - targets).square().mean()

# Pretraining loop
pretrain_optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_pretrain_iters = 1000

for i in range(num_pretrain_iters):
    pretrain_optimizer.zero_grad()
    loss = pretrain_loss(model, model.params, pts_pretrain)
    loss.backward()
    pretrain_optimizer.step()
    
    if i % 100 == 0:
        print(f"Pretrain Iter {i}: Loss= {loss.item():.6f}")

print("Pretraining completed!")

verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-3, -3, -1.5], dtype=torch.float64),
    bbox_max=torch.tensor([3, 3, 1.5], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig.display()


Pretrain Iter 0: Loss= 0.190957
Pretrain Iter 100: Loss= 0.084315
Pretrain Iter 200: Loss= 0.065986
Pretrain Iter 300: Loss= 0.012458
Pretrain Iter 400: Loss= 0.003530
Pretrain Iter 500: Loss= 0.001955
Pretrain Iter 600: Loss= 0.001085
Pretrain Iter 700: Loss= 0.000578
Pretrain Iter 800: Loss= 0.000311
Pretrain Iter 900: Loss= 0.000185
Pretraining completed!


Output()

### Main training loop

In [38]:
pts_eikonal = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
#pts_eikonal = torch.cat((pts_eikonal, pts_boundary))
pts_surface = pts_boundary

# Gauss-Newton weights:
# loss_weights = {"interface": 1.0, "eikonal": 0.001, "gauss_curvature": 1.0}
# Adam weights:
loss_weights = {"interface": 1.0, "eikonal": 0.1, "gauss_curvature": 1.0}

vals_gauss_curvature = torch.zeros(pts_surface.shape[0], dtype=torch.float64)
# vals_gauss_curvature = dome_curvature(pts_surface)

config = {
    "pts_boundary": pts_boundary,
    "pts_eikonal": pts_eikonal,
    "pts_surface": pts_surface,
    "vals_gauss_curvature": vals_gauss_curvature,
    "loss_weights": loss_weights,
    "regularization": 1e-6,
}

model = model.double()

In [39]:
model = model.double()
params = model.params
# Gauss-Newton:
optimizer = GaussNewtonNew(model, lr=1e-1, config=config)
# Adam:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for i in (pbar:=trange(1000)):
    optimizer.zero_grad()
    with torch.no_grad():
        loss_interface  = 0.5*model.f(params, pts_boundary).square().mean()
    
        loss_eikonal  = 0.5*model.r_eikonal(params, pts_eikonal).squeeze(1).square().mean()
    
        pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=1)
        pts_surface = sample_model_surface_newton(model, pts_surface)
        # pts_surface = pts_eikonal
        vals_gauss_curvature = torch.zeros(pts_surface.shape[0], dtype=torch.float64)
        # vals_gauss_curvature = dome_curvature(pts_surface)
        config["pts_surface"] = pts_surface
        config["vals_gauss_curvature"] = vals_gauss_curvature
        optim.config = config
        
        loss_gauss_curvature = 0.5*model.r_gauss_curvature(params, pts_surface, vals_gauss_curvature).squeeze(1).square().mean()
           
        loss = loss_weights["interface"] * loss_interface + loss_weights["eikonal"] * loss_eikonal + loss_weights["gauss_curvature"] * loss_gauss_curvature
        config["regularization"] = min(1e-4, loss)
        pbar.set_description(f"interface: {loss_interface.item():.2e} "
                                f"eikonal: {loss_eikonal.item():.2e} "
                                f"gauss curvature: {loss_gauss_curvature.item():.2e} "
                                f"{len(pts_surface)}"
                                )
    optimizer.step()

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [32]:
model = model.double()
params = model.params
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=18000)

for i in (pbar := trange(10000)):
    optimizer.zero_grad()
    
    # Sample points on the surface (no torch.no_grad here since we need gradients!)
    pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=1)
    pts_surface = sample_model_surface_newton(model, pts_surface)
    
    vals_gauss_curvature = torch.zeros(pts_surface.shape[0], dtype=torch.float64)
    # Compute losses
    loss_interface = 0.5 * model.f(params, pts_boundary).square().mean()
    loss_eikonal = 0.5 * model.r_eikonal(params, pts_eikonal).squeeze(1).square().mean()
    loss_gauss_curvature = 0.5 * model.r_gauss_curvature(params, pts_surface, vals_gauss_curvature).squeeze(1).square().mean()

    loss = (
        loss_weights["interface"] * loss_interface +
        loss_weights["eikonal"] * loss_eikonal +
        loss_weights["gauss_curvature"] * loss_gauss_curvature
    )

    # Logging
    pbar.set_description(
        f"interface: {loss_interface.item():.2e} "
        f"eikonal: {loss_eikonal.item():.2e} "
        f"gauss curvature: {loss_gauss_curvature.item():.2e} "
        f"{len(pts_surface)}"
    )

    # Backward and update
    loss.backward()
    optimizer.step()


  0%|          | 0/10000 [00:00<?, ?it/s]

In [34]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-1.2, -1.2, -0.3], dtype=torch.float64),
    bbox_max=torch.tensor([1.2, 1.2, 1.2], dtype=torch.float64),
    chunks=2
)


fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.01)
fig.display()

Output()

In [41]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-1.2, -1.2, -0.3], dtype=torch.float64),
    bbox_max=torch.tensor([1.2, 1.2, 1.2], dtype=torch.float64),
    chunks=2
)


fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.01)
fig.display()

Output()

### Visualize the result

In [103]:
verts, faces = get_mesh(
    model.float(), N=256, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

model.double()
mean_curvatures = model.r_mean_curvature(params, torch.tensor(verts, dtype=torch.float64)).squeeze(1).abs()

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)

color_map = k3d.basic_color_maps.Jet
color_range = [0, 0.005]

fig += k3d.mesh(
    verts, faces, 
    attribute=mean_curvatures.cpu().detach().numpy().astype(np.float32),
    color_range=color_range,
    color_map=color_map,
    side='double',
    flat_shading=False
)

fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

Output()

### LBFGS